# `RunnableSequence: RunnableSerializable[Input, Output]`

`RunnableSequence` runs multiple `Runnable` objects one after another.

The output of each step becomes the input of the next step.

It is commonly created using the `|` operator.

## Type Parameters

```python
Input # Input type accepted by the first Runnable
Output # Output type produced by the last Runnable
```

## Fields

```python
first: Runnable[Input, Any] # First Runnable in the sequence
middle: list[Runnable[Any, Any]] # Runnables executed between the first and last steps
last: Runnable[Any, Output] # Final Runnable in the sequence
name: str | None # Optional name used for tracing and debugging
```

## Constructor

```python
RunnableSequence(
    *steps: RunnableLike[Any, Any], # Ordered Runnable-like steps included in the sequence
    name: str | None = None, # Optional custom sequence name
    first: Runnable[Any, Any] | None = None, # Optional first Runnable
    middle: list[Runnable[Any, Any]] | None = None, # Optional middle Runnables
    last: Runnable[Any, Any] | None = None, # Optional final Runnable
) -> None # Initialize the RunnableSequence
```

The sequence must contain at least two steps.

Callable functions, generator functions, and mappings are automatically converted into suitable `Runnable` objects.

Nested `RunnableSequence` objects are flattened into one sequence.

## Property

### `steps`

Returns all Runnables in execution order.

```python
steps: list[Runnable[Any, Any]] # Ordered list containing first, middle, and last steps
```

## Overridden Properties and Methods

### `get_lc_namespace`

Returns the LangChain serialization namespace for Runnable objects.

### `is_lc_serializable`

Returns `True`, indicating that `RunnableSequence` supports LangChain serialization.

### `InputType`

Returns the input type of the first Runnable.

### `OutputType`

Returns the output type of the last Runnable.

### `get_input_schema`

Returns the sequence input schema based on the first relevant step.

It also supports schema changes introduced by assignment and passthrough steps.

### `get_output_schema`

Returns the sequence output schema based on the last relevant step.

It also supports schema changes introduced by assignment and passthrough steps.

### `config_specs`

Returns the unique configurable-field specifications collected from all steps.

### `get_graph`

Combines the graphs of all steps into one connected sequential execution graph.

### `__repr__`

Returns a pipeline-style representation of all steps.

### `__or__`

Appends another Runnable-like object to the sequence.

When another `RunnableSequence` is supplied, both sequences are flattened into one sequence.

### `__ror__`

Prepends another Runnable-like object to the sequence.

When another `RunnableSequence` is supplied, both sequences are flattened into one sequence.

### `invoke`

Synchronously executes every step in order.

Only the first step receives additional invocation keyword arguments.

### `ainvoke`

Asynchronously executes every step in order.

Only the first step receives additional invocation keyword arguments.

### `batch`

Processes multiple inputs by calling the batch method of each step in sequence.

When `return_exceptions=True`, failed inputs are excluded from later steps and their exceptions are restored in the final result list.

### `abatch`

Asynchronously processes multiple inputs by calling the asynchronous batch method of each step in sequence.

When `return_exceptions=True`, failed inputs are excluded from later steps and their exceptions are restored in the final result list.

### `transform`

Passes a synchronous input stream through every step in order.

A step without native transformation support may buffer its complete input before producing output.

### `stream`

Wraps one input as a synchronous iterator and streams it through the sequence.

### `atransform`

Passes an asynchronous input stream through every step in order.

A step without native asynchronous transformation support may delay downstream streaming.

### `astream`

Wraps one input as an asynchronous iterator and streams it through the sequence.

## Behaviour

- Steps execute from first to last.
- Each step receives the previous step's output.
- At least two steps are required.
- Nested sequences are flattened automatically.
- Batching uses each component's own batch implementation.
- Streaming is preserved when the sequence components support transformation.
- Streaming begins only after the last blocking component finishes.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableSequence # Import required classes

def add_five(number: int) -> int: # Define the first operation
    return number + 5 # Return the number after adding 5

def multiply_by_two(number: int) -> int: # Define the second operation
    return number * 2 # Return the number multiplied by 2

def subtract_three(number: int) -> int: # Define the third operation
    return number - 3 # Return the number after subtracting 3

add = RunnableLambda(add_five) # Convert the first function into a Runnable
multiply = RunnableLambda(multiply_by_two) # Convert the second function into a Runnable
subtract = RunnableLambda(subtract_three) # Convert the third function into a Runnable

In [ ]:
# Using the | operator
sequence_1 = add | multiply | subtract # Create a RunnableSequence using the pipe operator

result_1 = sequence_1.invoke(10) # Execute all Runnables sequentially

print(result_1) # Display the result

In [ ]:
# Using positional constructor arguments
sequence_2 = RunnableSequence( # Create RunnableSequence directly
    add, # Set the first step
    multiply, # Set the second step
    subtract, # Set the third step
)

result_2 = sequence_2.invoke(10) # Execute all Runnables sequentially

print(result_2) # Display the result

In [ ]:
# Using first, middle, and last
sequence_3 = RunnableSequence( # Create RunnableSequence using named fields
    first=add, # Set the first Runnable
    middle=[multiply], # Set the middle Runnables
    last=subtract, # Set the final Runnable
)

result_3 = sequence_3.invoke(10) # Execute all Runnables sequentially

print(result_3) # Display the result

In [ ]:
# Using pipe()
sequence_4 = add.pipe( # Create a sequence starting with add
    multiply, # Add the second Runnable
    subtract, # Add the third Runnable
)

result_4 = sequence_4.invoke(10) # Execute all Runnables sequentially

print(result_4) # Display the result